# 04 Video Evaluation

Evaluate the trained CNN-LSTM model on the test set and perform inference on sample videos.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import models, transforms
from pathlib import Path
import pandas as pd
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_curve, auc, classification_report

# Config
IMG_SIZE = 224
NUM_FRAMES = 16
MODEL_PATH = Path("../../models/video/best_weights.pth")
TEST_DATA_ROOT = Path("../../data/splits/videos/test")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Model Definition & Loading

In [ ]:
class DeepfakeVideoModel(nn.Module):
    def __init__(self, hidden_dim=256):
        super(DeepfakeVideoModel, self).__init__()
        backbone = models.efficientnet_b0(weights=None) # No need for pretrained if loading weights
        self.feature_extractor = nn.Sequential(*list(backbone.children())[:-1])
        self.lstm = nn.LSTM(input_size=1280, hidden_size=hidden_dim, num_layers=1, batch_first=True)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )
        
    def forward(self, x):
        batch_size, time_steps, C, H, W = x.size()
        x = x.view(batch_size * time_steps, C, H, W)
        features = self.feature_extractor(x)
        features = features.view(batch_size * time_steps, -1)
        features = features.view(batch_size, time_steps, -1)
        lstm_out, (h_n, c_n) = self.lstm(features)
        out = self.classifier(h_n[-1])
        return out

model = DeepfakeVideoModel().to(device)
if MODEL_PATH.exists():
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    model.eval()
    print("Best model weights loaded.")
else:
    print(f"Warning: {MODEL_PATH} not found.")

## 2. Test Set Evaluation

In [ ]:
# Re-implementing Dataset for test set
class VideoFrameDataset(torch.utils.data.Dataset):
    def __init__(self, root_path, num_frames=16, transform=None):
        self.data = []
        for label in ["real", "fake"]:
            folder = root_path / label
            for video_path in folder.glob("*.mp4"):
                self.data.append({"path": str(video_path), "label": 1 if label == "fake" else 0})
        self.num_frames = num_frames
        self.transform = transform
        
    def __len__(self): return len(self.data)
    
    def _get_frames(self, video_path):
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total_frames <= 0: cap.release(); return torch.zeros((self.num_frames, 3, IMG_SIZE, IMG_SIZE))
        indices = np.linspace(0, total_frames - 1, self.num_frames).astype(int)
        frames = []
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if not ret: frame = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
            else: frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = Image.fromarray(frame)
            if self.transform: frame = self.transform(frame)
            frames.append(frame)
        cap.release()
        return torch.stack(frames)
    
    def __getitem__(self, index):
        item = self.data[index]
        return self._get_frames(item["path"]), item["label"]

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_dataset = VideoFrameDataset(TEST_DATA_ROOT, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

all_preds, all_labels = [], []
with torch.no_grad():
    for frames, labels in test_loader:
        frames = frames.to(device)
        outputs = model(frames)
        probs = torch.sigmoid(outputs).cpu().numpy()
        all_preds.extend(probs)
        all_labels.extend(labels.numpy())

all_preds = np.array(all_preds).flatten()
all_labels = np.array(all_labels)
binary_preds = (all_preds >= 0.5).astype(int)

print(classification_report(all_labels, binary_preds, target_names=["Real", "Fake"]))

## 3. Confusion Matrix & ROC Curve

In [ ]:
cm = confusion_matrix(all_labels, binary_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Real", "Fake"], yticklabels=["Real", "Fake"])
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

fpr, tpr, _ = roc_curve(all_labels, all_preds)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color="darkorange", lw=2, label=f"ROC curve (AUC = {roc_auc:.2f})")
plt.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Receiver Operating Characteristic")
plt.legend(loc="lower right")
plt.show()

## 4. Single Video Inference

In [ ]:
def predict_video(video_path, model, num_frames=16):
    model.eval()
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0: return None
    
    indices = np.linspace(0, total_frames - 1, num_frames).astype(int)
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret: frame = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        else: frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = Image.fromarray(frame)
        frame = transform(frame)
        frames.append(frame)
    cap.release()
    
    frames_tensor = torch.stack(frames).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(frames_tensor)
        prob = torch.sigmoid(output).item()
        
    return "FAKE" if prob >= 0.5 else "REAL", prob

# Example usage:
# label, score = predict_video("path/to/video.mp4", model)
# print(f"Prediction: {label} ({score:.4f})")